In [1]:
import pandas as pd
#importar el dataset y generar un summary estadistico, ademas formatear la columnas de fechas
df = pd.read_csv('PdB_Dumps_Attrbs\ParsedObserverLogs_wDates.csv')

df.describe()

,Wind,Sea State,Ambient Noise,Water Depth,Swell Noise,Direccion
count,131.000000,131.000000,131.000000,131.000000,131.000000,129.000000
mean,15.778626,1.562341,8.484097,1262.672519,0.473282,177.906977
std,5.235295,0.427593,4.307204,179.200917,0.501202,90.326443
min,6.500000,0.500000,0.666667,0.000000,0.000000,90.000000
25%,12.500000,1.250000,5.000000,1235.425000,0.000000,90.000000
50%,15.000000,1.500000,7.500000,1268.650000,0.000000,90.000000
75%,17.500000,2.000000,11.000000,1345.500000,1.000000,270.000000
max,32.500000,3.250000,23.000000,1565.050000,1.000000,270.000000


Vamos a trabajar con la probabilidad marginal de Swell Noise, eso se calcula simplemente haciendo el conteo de todos los valores en el subset A y B, para luego normalizar sobre el numero de valores totales

Sea $\Omega$ el espacio muestral completo. Lo particionamos en dos eventos mutuamente excluyentes y exhaustivos:

$$\Omega = A \cup B, \quad A \cap B = \emptyset$$

donde $A_0 = \{\text{Swell Noise} = 1\}$ y $A_1 = \{\text{Swell Noise} = 0\}$

Por lo tanto: $P(A) + P(B) = 1$


A esto le llamamos creencias inciales o priors...
P(A) y P(B)

In [2]:
A_0 = df['Swell Noise'].sum()
A_1 = (df['Swell Noise'] == 0).sum()

Prob_SwellNoise = A_0 / (A_0 + A_1)
Prob_NoSwellNoise = 1 - Prob_SwellNoise

print(f"Probabilidad de Swell Noise: {Prob_SwellNoise}")
print(f"Probabilidad de No Swell Noise: {Prob_NoSwellNoise}")

Probabilidad de Swell Noise: 0.4732824427480916
Probabilidad de No Swell Noise: 0.5267175572519084


Ahora para hacer uso de la probabilidad condicional debemos construir un conjunto B, el cual contendrá eventos que pueden ser una discretización de cualquier atributo que prefieras. 
De modo que ahora calcularemos P(B|A_1), osea la probabilidad de observar B dado que ya ocurrió A_1. Entendiendo A_1 como nuestro nuevo universo. 

El conjunto B, va constar de n eventos, con n = numero de bins de cada atributo.

In [3]:
#Creamos el conjunto B, discretizando cualquier atributo

def discretizar_atributo(df, atributo, bins):
    """
    Agrega al DataFrame una columna 'Bin' con el atributo discretizado
    y retorna también la información de rangos de cada bin.
    """
    labels = [f'Bin_{i}' for i in range(bins)]

    resultado, bin_edges = pd.qcut(
        df[atributo],
        q=bins,
        labels=labels,
        retbins=True
    )

    df_resultado = df.copy()
    df_resultado['Bin'] = resultado

    bin_info = pd.DataFrame({
        'rango_min': bin_edges[:-1],
        'rango_max': bin_edges[1:]
    }, index=labels)
    bin_info.index.name = 'bin'

    return df_resultado, bin_info
ambient_binned, ambient_bin_info = discretizar_atributo(df, 'Ambient Noise', 4)
sea_state_binned, sea_state_bin_info = discretizar_atributo(df, 'Sea State', 4)

In [31]:
count_AnB_1 = ( (df_binned['Bin'] == 'Bin_0') & (df_binned['Swell Noise'] == 1) ).sum()
count_AnB_2 = ( (df_binned['Bin'] == 'Bin_1') & (df_binned['Swell Noise'] == 1) ).sum()
count_AnB_3 = ( (df_binned['Bin'] == 'Bin_2') & (df_binned['Swell Noise'] == 1) ).sum()
count_AnB_4 = ( (df_binned['Bin'] == 'Bin_3') & (df_binned['Swell Noise'] == 1) ).sum()

Para cada escenario A, que tan probable es observar B 
$ P(B|A) $

O sea, ya que miré que A_0 = Swell y A_1 = No Swell, que tan probable es observar los valores del Bin_n de cierto atributo.

Tengo una causa A, quiero ver de entre todos los posibles efectos B(eventos en B), cuál es probable que suceda.
El oleaje en este caso es el que ocasiona el efecto de observar los valores del bin_n

In [34]:
#P(B|A) = Count(A and B) / Count(A)

P_B0_given_A = count_AnB_1 / A_0
P_B1_given_A = count_AnB_2 / A_0
P_B2_given_A = count_AnB_3 /A_0
P_B3_given_A = count_AnB_4 / A_0
print(f"Probabilidad de Bin_0 dado Swell Noise: {P_B0_given_A}")
print(f"Probabilidad de Bin_1 dado Swell Noise: {P_B1_given_A}")
print(f"Probabilidad de Bin_2 dado Swell Noise: {P_B2_given_A}")
print(f"Probabilidad de Bin_3 dado Swell Noise: {P_B3_given_A}")

Probabilidad de Bin_0 dado Swell Noise: 0.1774193548387097
Probabilidad de Bin_1 dado Swell Noise: 0.1935483870967742
Probabilidad de Bin_2 dado Swell Noise: 0.25806451612903225
Probabilidad de Bin_3 dado Swell Noise: 0.3709677419354839


In [36]:
#Ahora calcularemos para Swell == 0
P_B0_given_A1 = ( (df_binned['Bin'] == 'Bin_0') & (df_binned['Swell Noise'] == 0) ).sum() / A_1
P_B1_given_A1 = ( (df_binned['Bin'] == 'Bin_1') & (df_binned['Swell Noise'] == 0) ).sum() / A_1
P_B2_given_A1 = ( (df_binned['Bin'] == 'Bin_2') & (df_binned['Swell Noise'] == 0) ).sum() / A_1
P_B3_given_A1 = ( (df_binned['Bin'] == 'Bin_3') & (df_binned['Swell Noise'] == 0) ).sum() / A_1

print(f"Probabilidad de Bin_0 dado No Swell Noise: {P_B0_given_A1}")
print(f"Probabilidad de Bin_1 dado No Swell Noise: {P_B1_given_A1}")
print(f"Probabilidad de Bin_2 dado No Swell Noise: {P_B2_given_A1}")
print(f"Probabilidad de Bin_3 dado No Swell Noise: {P_B3_given_A1}")

Probabilidad de Bin_0 dado No Swell Noise: 0.4057971014492754
Probabilidad de Bin_1 dado No Swell Noise: 0.2753623188405797
Probabilidad de Bin_2 dado No Swell Noise: 0.21739130434782608
Probabilidad de Bin_3 dado No Swell Noise: 0.10144927536231885


A pesar de que la formula general para probabilidad condicional es P(A|B) = P(AnB)/P(B), cuando hablamos de modelar fenomenos secuenciales o físicos, pues primero tendremos la causa (A), el fenomeno físico real y luego el efecto, la medición del instrumento (B).

En estos casos lo mejor es depejar la formula y trabajar con P(AnB) = P(B)*P(A|B), no te preocupes si quieres hacer P(B|A), ya que la intersección de AnB = BnA. Entonces; 

$P(A \cap B) = P(B)*P(A|B) = P(A)*P(B|A)$


Pasando a la regla de la cadena o multiplication rule, sirve para calcular la probabiliad de que varios eventos sucedan a la vez; 
Sea A un conjunto con 2 eventos {Swell = 1, Swell = 0}
Sea B un conjunto con 4 eventos {Amient Noise = bin1, 2, 3, 4}
Sea C un conjunto con 4 eventos {Sea State = bin1, 2,3,4}

In [50]:
#Probabilidad conjunta de A, B y C, P(A=Swell=1,B=bin1,C=bin3) = P(A|B,C) * P(B|C) * P(C)

N = len(df)
N_c = (sea_state_binned['Bin'] == 'Bin_3').sum()

Prob_C = N_c / N
print(f"Probabilidad de C (Sea State Bin_3): {Prob_C}")

Probabilidad de C (Sea State Bin_3): 0.03816793893129771


In [52]:
Prob_B_dado_C = ((sea_state_binned['Bin'] == 'Bin_3') & (ambient_binned['Bin'] == 'Bin_1')).sum() / N_c
print(f"Probabilidad de B (Ambient Noise Bin_1) dado C (Sea State Bin_3): {Prob_B_dado_C}")

Probabilidad de B (Ambient Noise Bin_1) dado C (Sea State Bin_3): 0.2


In [53]:
Prob_A_dado_B_C = ((sea_state_binned['Bin'] == 'Bin_3') & (ambient_binned['Bin'] == 'Bin_1') & (df['Swell Noise'] == 1)).sum() / ((sea_state_binned['Bin'] == 'Bin_3') & (ambient_binned['Bin'] == 'Bin_1')).sum()
print(f"Probabilidad de A (Swell Noise=1) dado B (Ambient Noise Bin_1) y C (Sea State Bin_3): {Prob_A_dado_B_C}")

Probabilidad de A (Swell Noise=1) dado B (Ambient Noise Bin_1) y C (Sea State Bin_3): 1.0


In [54]:
Prob_A_B_C = Prob_A_dado_B_C * Prob_B_dado_C * Prob_C
print(f"Probabilidad conjunta de A, B y C: {Prob_A_B_C*100:.4f}%")

Probabilidad conjunta de A, B y C: 0.7634%


De aquí podemos constatar que la probabilidad conjunta de observar swell noise, en un mar con mediciones de ambient noise bajas y sea state altas es super inestable debido a los problemas de muestremo. Igual por la condiciones físicas, un ambient noise entonces se correlaciona a un sea state alto.

Otra forma de llegar a esta probabilidad;
$P(A \cap B \cap C) = \frac{\text{count}(C)}{N} \cdot \frac{\text{count}(B \cap C)}{\text{count}(C)} \cdot \frac{\text{count}(A \cap B \cap C)}{\text{count}(B \cap C)} $

Y esta claro que podemos ir directo a la probabilidad, haciendo; 

$ P(A \cap B \cap C) = \frac{\text{count}(A \cap B \cap C)}{N} $

In [62]:
def regla_cadena(df, atributos_bins, target_col, target_val):
    """
    atributos_bins: list of (col, bin_label, n_bins)
    El último elemento actúa como condición base P(C); el orden define la cadena.
    """
    N = len(df)

    # Discretizar cada atributo y construir su máscara booleana
    mascaras = []
    for col, bin_label, n_bins in atributos_bins:
        binned = pd.qcut(df[col], q=n_bins, labels=[f'Bin_{i}' for i in range(n_bins)], duplicates='drop')
        mascaras.append((col, bin_label, binned == bin_label))

    target_mask = df[target_col] == target_val

    # Aplicar la cadena de atrás hacia adelante: P(C) → P(B|C) → P(A|B,C) → P(target|...)
    prob_cadena = 1.0
    cond_acum = pd.Series(True, index=df.index)

    for col, bin_label, mascara in reversed(mascaras):
        p = (cond_acum & mascara).sum() / cond_acum.sum()
        prob_cadena *= p
        cond_acum &= mascara
        print(f"  P({col}={bin_label} | condiciones anteriores) = {p:.6f}")

    p_target = (cond_acum & target_mask).sum() / cond_acum.sum()
    prob_cadena *= p_target
    print(f"  P({target_col}={target_val} | todas las condiciones) = {p_target:.6f}")

    # Verificación: cálculo directo count(A∩B∩C) / N
    mascara_total = target_mask.copy()
    for _, _, m in mascaras:
        mascara_total &= m
    prob_directa = mascara_total.sum() / N

    print(f"\nRegla de cadena : {prob_cadena:.6f}  ({prob_cadena*100:.4f}%)")
    print(f"Cálculo directo : {prob_directa:.6f}  ({prob_directa*100:.4f}%)")

    return prob_cadena, prob_directa

# Ejemplo: P(Swell=1 ∩ AmbientNoise=Bin_1 ∩ SeaState=Bin_3)
regla_cadena(
    df,
    atributos_bins=[('Ambient Noise', 'Bin_3', 4), ('Sea State', 'Bin_3', 4)],
    target_col='Swell Noise',
    target_val=1
)


  P(Sea State=Bin_3 | condiciones anteriores) = 0.038168
  P(Ambient Noise=Bin_3 | condiciones anteriores) = 0.800000
  P(Swell Noise=1 | todas las condiciones) = 0.750000

Regla de cadena : 0.022901  (2.2901%)
Cálculo directo : 0.022901  (2.2901%)


(np.float64(0.02290076335877863), np.float64(0.022900763358778626))

In [4]:
def cadena_directa(df, atributos_bins, target_col, target_val):
    """
    Calcula la probabilidad directa de la intersección de eventos.
    """
    N = len(df)

        # Construir la máscara total para la intersección
    mascara_total = df[target_col] == target_val
    for col, bin_label, n_bins in atributos_bins:
        binned = pd.qcut(df[col], q=n_bins, labels=[f'Bin_{i}' for i in range(n_bins)], duplicates='drop')
        mascara_total &= (binned == bin_label)

    #Informe de los conteos por bin
    for col, bin_label, n_bins in atributos_bins:
        binned = pd.qcut(df[col], q=n_bins, labels=[f'Bin_{i}' for i in range(n_bins)], duplicates='drop')
        count = (binned == bin_label).sum()
        print(f"Conteo de {col}={bin_label}: {count}")

    prob_directa = mascara_total.sum() / N
    print(f"Probabilidad directa de la intersección: {prob_directa:.6f}  ({prob_directa*100:.4f}%)")

cadena_directa(df,
    atributos_bins=[('Ambient Noise', 'Bin_1', 2), ('Sea State', 'Bin_1', 2)],
    target_col='Swell Noise',
    target_val=0
)

Conteo de Ambient Noise=Bin_1: 61
Conteo de Sea State=Bin_1: 51
Probabilidad directa de la intersección: 0.129771  (12.9771%)
